```mermaid
flowchart TD
    U([User])

    APP[Application]

    DB[(DB)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U

In [ ]:
def rag(question):
    context = search(question)
    prompt = build_prompt(question,context)
    return llm(prompt)

### Fetch Data

In [1]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
course_raw = response.json()

course_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [2]:
documents = []
docs_url = "https://datatalks.club/faq"
for course in course_raw:
    course_url = f"{docs_url}{course['path']}"
    response = requests.get(course_url)
    response.raise_for_status()
    data = response.json()
    documents.extend(data)

len(documents)

1342

In [3]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

### Build Search Index

In [4]:
from minsearch import Index
index = Index(
    text_fields=['section','answer','question'],
    keyword_fields=['course'],
)

index.fit(documents)

### Boosting Fields

In [7]:
question = "I just discovered the course. Can I join now?"

In [11]:
boost_result = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    num_results=6
)

boost_result[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

### Filtering Field

In [14]:
filter_result = index.search(
    question,
    filter_dict={'course': 'data-engineering-zoomcamp'},
    num_results=5
)

len(filter_result)

5

### Wrapping in a funtion

In [17]:
def search(question,course='llm-zoomcamp'):
    boost_dict={'question' : 2.0, 'section':0.5}
    filter_dict={'course' : course}
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

search_result=search(question)
search_result

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

### Building the prompt

In [24]:
SYSTEM_PROMPT= """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [36]:
USER_PROMPT_TEMPLATE="""
Question :
{question}
    
Context : 
{context}
"""

In [26]:
def build_context(search_result):
    lines = []
    for doc in search_result:
        lines.append(doc['section'])
        lines.append("Q: " + doc['question'])
        lines.append("A: " + doc['answer'])
        lines.append("")
    return '\n'.join(lines).strip()

In [27]:
context = build_context(search_result)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [37]:
def build_prompt(question,search_result):
    context = build_context(search_result)
    prompt= USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context,
    )
    return prompt

prompt = build_prompt(question,search_result)
print(prompt)


Question :
I just discovered the course. Can I join now?

Context : 
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your pro

### LLM Generation

In [43]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
load_dotenv()
gemini_client = genai.Client()

In [44]:
def llm(system_prompt,user_prompt,model='gemini-2.5-flash'):
    response = gemini_client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt, 
            temperature=0.0 
        )
    )
    return response.text

In [45]:
def llm_rag(question,model='gemini-2.5-flash'):
    search_result = search(question)
    user_prompt = build_prompt(question,search_result) #build context -> question + context = prompt
    answer = llm(SYSTEM_PROMPT,user_prompt)
    return answer


In [47]:
print(llm_rag("How do i get a certificate?"))

To get a certificate, you need to:

1.  Finish the course with a "live" cohort, as certificates are not awarded for self-paced mode.
2.  Pass the Capstone project. This involves submitting your project and then peer-reviewing 3 other capstone projects. Peer-reviewing can only be done when the course is actively running.
3.  Ensure your official name (as it appears on identification documents) is updated in your course profile, as this name will appear on your certificate.

Missing homework does not prevent you from getting a certificate, but passing the Capstone project is mandatory.
